# Time Delay with AGNSF

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import agnsf

rng = np.random.default_rng(13)

t = np.arange(0.0, 200.0, 1.0)

# Continuum light curve (i.e., driving LC)
cont = np.cumsum(rng.normal(0.0, 1.0, t.size))
e1 = np.full_like(t, 0.3)

# Response = continuum delayed by lag_true + noise.
lag_true = 15.0
e2_scale = 0.5
resp = np.interp(t - lag_true, t, cont) + rng.normal(0.0, e2_scale, t.size)
e2 = np.full_like(t, e2_scale)


plt.errorbar(t, cont, e1, ls='-', lw=1, marker='.')
plt.errorbar(t, resp, e2, ls='none',    marker='.')

## DCF, centroid estimate, FR-RSS uncertainty

`agnsf.timedelay.lag` returns a `LagResult` with the selected lag, both peak/centroid estimates, the FR-RSS interval, and the full CCF curve.


In [ ]:
res = agnsf.timedelay.lag(
    t, cont, e1, t, resp, e2,
    lag_range=(-50.0, 50.0), step=1.0,
    method="dcf", estimate="centroid",
    uncertainty="fr_rss", n_realizations=500, seed=0,
)

print("lag       :", res.lag)
print("peak      :", res.lag_peak, " centroid:", res.lag_centroid)
print("fr-rss    :", res.lower, res.upper)


## ICCF variant

Switch to the interpolated cross-correlation function with a peak estimate (no uncertainty, for speed).


In [ ]:
res_iccf = agnsf.timedelay.lag(
    t, cont, e1, t, resp, e2,
    lag_range=(-50.0, 50.0), step=1.0,
    method="iccf", estimate="peak", uncertainty=None,
)

print("iccf lag  :", res_iccf.lag)


## Plot the correlation function


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 3))
plt.plot(res.tau, res.ccf, lw=2.5)
plt.axvline(res.lag, color="tab:red", ls="--", label=f"lag = {res.lag:.1f}")
plt.xlabel("lag")
plt.ylabel("CCF")
plt.legend()
plt.show()
